## Numerical Analysis-Assignment

### Task 1 (2 + 3 + 10 + 3 + 2 P)
Solve a two-dimensional Poisson problem using the finite difference method. Consider the following boundary value problem

\begin{aligned}
-\Delta u &= f \quad \text{ in }\Omega=(0,1) \times (0,1)\\
u&=0 \quad \text{ on }\partial\Omega
\end{aligned}

with $f(x,y) = 2 \pi^2 \sin(\pi x) \sin(\pi y)$.

In this task, as in the lecture, $h$ is the distance between two neighboring grid points. Then, $\ell-1$ is the number of grid points of $\Omega_h$ in each dimension, $1/h = \ell$, and the number of grid points is $(\ell -1)^2$. The discrete domain (grid) $\Omega_h$ is then given by

\begin{equation*}
(x_i, y_j) \in \Omega_h \quad i,j =1,\dots,\ell-1.
\end{equation*}

1.  Verify with pen and paper that $u(x,y) = \sin(\pi x) \sin(\pi y)$ is the solution of the above boundary value problem. You may write the solution here.


We verify that $u(x,y) = \sin(\pi x)\sin(\pi y)$ solves the boundary value problem.

**Interior equation.** Compute the Laplacian:

$$
\frac{\partial^2 u}{\partial x^2} = -\pi^2 \sin(\pi x)\sin(\pi y), \qquad
\frac{\partial^2 u}{\partial y^2} = -\pi^2 \sin(\pi x)\sin(\pi y)
$$

$$
\Rightarrow \quad -\Delta u = -\!\left(\frac{\partial^2 u}{\partial x^2}+\frac{\partial^2 u}{\partial y^2}\right) = 2\pi^2\sin(\pi x)\sin(\pi y) = f(x,y). \quad \checkmark
$$

**Boundary condition.** On $\partial\Omega$ we have $x\in\{0,1\}$ or $y\in\{0,1\}$.
Since $\sin(0)=\sin(\pi)=0$ it follows that $u=0$ on $\partial\Omega$. $\checkmark$

---


2. Discretize the problem using the finite difference method and write the resulting system $A_h \boldsymbol u_h = \boldsymbol q_h$ for $h = 1/4$ with pen and paper. You may write the solution here.


With $h=\tfrac14$ we have $\ell=4$, giving $n = \ell-1 = 3$ interior grid points per dimension and $N = n^2 = 9$ unknowns total.

**Grid points** $x_i = ih,\; y_j = jh$, $\; i,j \in \{1,2,3\}$.

The standard five-point stencil approximates $-\Delta u$ at $(x_i,y_j)$:

$$
\frac{1}{h^2}\bigl(-u_{i-1,j} - u_{i+1,j} - u_{i,j-1} - u_{i,j+1} + 4u_{i,j}\bigr) = f_{i,j}.
$$

**Lexicographic ordering** (x-index varies fastest):

$$
k = (j-1)\cdot 3 + i, \qquad k=1,\dots,9
$$

|$k$|$(i,j)$|$(x_i,y_j)$|
|:-:|:-----:|:---------:|
|1|(1,1)|$(¼,¼)$|
|2|(2,1)|$(½,¼)$|
|3|(3,1)|$(¾,¼)$|
|4|(1,2)|$(¼,½)$|
|5|(2,2)|$(½,½)$|
|6|(3,2)|$(¾,½)$|
|7|(1,3)|$(¼,¾)$|
|8|(2,3)|$(½,¾)$|
|9|(3,3)|$(¾,¾)$|

**System matrix** $A_h = \dfrac{1}{h^2}(T_3 \otimes I_3 + I_3 \otimes T_3)$ with $h=\tfrac14$, i.e.\ $\tfrac{1}{h^2}=16$:

$$
A_h = 16\begin{pmatrix}
B & -I & 0 \\
-I & B & -I \\
0 & -I & B
\end{pmatrix}, \qquad
B = \begin{pmatrix}4&-1&0\\-1&4&-1\\0&-1&4\end{pmatrix}, \quad
I = I_3.
$$

Explicitly:

$$
A_h = 16\begin{pmatrix}
 4&-1& 0&-1& 0& 0& 0& 0& 0\\
-1& 4&-1& 0&-1& 0& 0& 0& 0\\
 0&-1& 4& 0& 0&-1& 0& 0& 0\\
-1& 0& 0& 4&-1& 0&-1& 0& 0\\
 0&-1& 0&-1& 4&-1& 0&-1& 0\\
 0& 0&-1& 0&-1& 4& 0& 0&-1\\
 0& 0& 0&-1& 0& 0& 4&-1& 0\\
 0& 0& 0& 0&-1& 0&-1& 4&-1\\
 0& 0& 0& 0& 0&-1& 0&-1& 4
\end{pmatrix}
$$

**Right-hand side.** Using $f(x,y)=2\pi^2\sin(\pi x)\sin(\pi y)$ and $\sin(\pi/4)=\tfrac{\sqrt{2}}{2}$, $\sin(\pi/2)=1$:

$$
\boldsymbol{q}_h = \pi^2
\begin{pmatrix}1\\\sqrt{2}\\1\\\sqrt{2}\\2\\\sqrt{2}\\1\\\sqrt{2}\\1\end{pmatrix}
$$

---



3.  Implement your discretization for general $h \le \tfrac12$ that satsifies that $1/h \in \mathbb{N}$. You may use the code structure below, but you can also change it.


```python
import numpy as np
import matplotlib
matplotlib.use('Agg')                  # ← must come BEFORE importing pyplot
import matplotlib.pyplot as plt
from matplotlib import cm
import scipy.sparse as sp
import scipy.sparse.linalg as spla


# ── Analytical solution and source term ───────────────────────────────────────

def u_exact(x, y):
    return np.sin(np.pi * x) * np.sin(np.pi * y)

def f(x, y):
    return 2.0 * np.pi**2 * np.sin(np.pi * x) * np.sin(np.pi * y)


# ── Matrix and RHS assembly ────────────────────────────────────────────────────

def create_matrix(h):
    """
    Build A_h = (1/h²) * (T ⊗ I + I ⊗ T)
    where T is the 1-D tridiagonal FD matrix.
    This Kronecker construction is correct for ALL valid h.
    """
    n = int(round(1.0 / h)) - 1   # interior points per dimension
    inv_h2 = 1.0 / h**2

    # 1-D tridiagonal: diag(2) + off-diag(-1)
    T = sp.diags([-1, 2, -1], [-1, 0, 1], shape=(n, n), format='csr', dtype=float)

    I = sp.eye(n, format='csr')

    # 2-D operator via Kronecker sum
    A = inv_h2 * (sp.kron(I, T, format='csr') + sp.kron(T, I, format='csr'))
    return A


def create_right_hand_side(h):
    """
    Assemble the RHS vector q_h = f evaluated at interior grid points.
    """
    n = int(round(1.0 / h)) - 1
    b = np.zeros(n * n)
    for j in range(n):          # y-index
        for i in range(n):      # x-index
            x = (i + 1) * h
            y = (j + 1) * h
            b[j * n + i] = f(x, y)
    return b


# ── Error computation ──────────────────────────────────────────────────────────

def compute_error(u_numerical, h):
    """
    Discrete L²-norm (eq. 1.18, Knabner/Angermann) of the error
    ||u_exact - u_h||_{L²_h} = h * sqrt( sum_{i,j} |e_{i,j}|^2 ).
    """
    n = int(round(1.0 / h)) - 1
    err = np.zeros(n * n)
    for j in range(n):
        for i in range(n):
            x = (i + 1) * h
            y = (j + 1) * h
            err[j * n + i] = u_exact(x, y) - u_numerical[j * n + i]
    return h * np.sqrt(np.sum(err**2))


# ── Plot ───────────────────────────────────────────────────────────────────────

def plot_solution(h, u_numerical):
    """
    3-D surface plots of the numerical and analytical solutions.
    """
    n = int(round(1.0 / h)) - 1

    # Embed interior values into full grid (with boundary zeros)
    u_grid = np.zeros((n + 2, n + 2))
    for j in range(n):
        for i in range(n):
            u_grid[j + 1, i + 1] = u_numerical[j * n + i]

    x = np.linspace(0, 1, n + 2)
    y = np.linspace(0, 1, n + 2)
    X, Y = np.meshgrid(x, y)
    U_anal = u_exact(X, Y)

    fig, axes = plt.subplots(
        1, 2, figsize=(12, 5),
        subplot_kw={'projection': '3d'}
    )

    axes[0].plot_surface(X, Y, u_grid, cmap=cm.viridis)
    axes[0].set_title(f'Numerical solution  (h = 1/{int(round(1/h))})')
    axes[0].set_xlabel('x'); axes[0].set_ylabel('y'); axes[0].set_zlabel('u')

    axes[1].plot_surface(X, Y, U_anal, cmap=cm.plasma)
    axes[1].set_title('Analytical solution')
    axes[1].set_xlabel('x'); axes[1].set_ylabel('y'); axes[1].set_zlabel('u')

    plt.tight_layout()
    plt.savefig('poisson_solution.png', dpi=150, bbox_inches='tight')
    #plt.show()


# ── EOC study ─────────────────────────────────────────────────────────────────

hs     = [1.0 / 2**k for k in range(1, 8)]   # h = 1/2, 1/4, ..., 1/128
errors = []

print(f"{'h':>10}  {'||e||_h':>14}")
print("-" * 28)
for h in hs:
    A   = create_matrix(h)
    b   = create_right_hand_side(h)
    u_h = spla.spsolve(A, b)
    err = compute_error(u_h, h)
    errors.append(err)
    print(f"1/{int(round(1/h)):<6}   {err:.6e}")

print(f"\n{'h':>10}  {'EOC':>8}")
print("-" * 22)
for k in range(1, len(errors)):
    eoc = np.log(errors[k-1] / errors[k]) / np.log(2.0)
    print(f"1/{int(round(1/hs[k])):<6}   {eoc:.4f}")

# ── Solution plot for h = 1/64 ────────────────────────────────────────────────
h_plot  = 1.0 / 64
A_plot  = create_matrix(h_plot)
b_plot  = create_right_hand_side(h_plot)
u_plot  = spla.spsolve(A_plot, b_plot)
plot_solution(h_plot, u_plot)
```

---



4. Calculate the experimental order of convergence (EOC) of your discretization using $h = \tfrac12, \tfrac14, \tfrac18, \tfrac{1}{16}, \dots$ by evaluating the discrete $L^2$-norm (equation (1.18) / page 27 in the Knabner/Angermann book, see Google Drive) of the difference between your analytical and the numerical solution evaluated in the grid points. What is the convergence rate $p$ of your discretization (as in $O(h^p)$)?

\begin{equation*}
 \text{EOC} = \frac{\log\left(\frac{||u^\text{analytical} -u^\text{numerical}_{h}||}{||u^\text{analytical} -u^\text{numerical}_{h/2}||}\right)}{\log(2)}
\end{equation*}



The discrete $L^2$-norm used is:

$$
\|e_h\|_{L^2_h} = h\sqrt{\sum_{i,j=1}^{n} \bigl|u(x_i,y_j) - u^h_{i,j}\bigr|^2}
$$

| $h$ | $\|e_h\|_{L^2_h}$ | EOC |
|:---:|:-----------------:|:---:|
| $1/2$   | $1.16850275 \times 10^{-1}$ | —      |
| $1/4$   | $2.65146438 \times 10^{-2}$ | 2.1398 |
| $1/8$   | $6.47537336 \times 10^{-3}$ | 2.0338 |
| $1/16$  | $1.60948222 \times 10^{-3}$ | 2.0084 |
| $1/32$  | $4.01788840 \times 10^{-4}$ | 2.0021 |
| $1/64$  | $1.00410905 \times 10^{-4}$ | 2.0005 |
| $1/128$ | $2.51004580 \times 10^{-5}$ | 2.0001 |

**The convergence rate is $p = 2$**, i.e. the error behaves as $\mathcal{O}(h^2)$.

The EOC at $h=1/2 \to 1/4$ is slightly above 2 (2.14) because the coarsest grid $h=1/2$
has only **1 interior point**, so the asymptotic regime has not been reached yet.
From $h=1/8$ onward the EOC settles cleanly to 2.000, confirming the expected
second-order accuracy of the five-point stencil.

---

5. Use matplotlib to plot the solution for $h = \tfrac{1}{64}$.

The code in Part 3 calls `plot_solution(1/64, u_plot)` and saves `poisson_solution.png`.
The left panel shows the numerical solution; the right panel shows $u(x,y)=\sin(\pi x)\sin(\pi y)$.
The two surfaces are visually indistinguishable, confirming the accuracy of the method.

![Poisson solution h=1/64](./poisson_solution.png)

In [7]:
import numpy as np
import matplotlib
matplotlib.use('Agg')                  # ← must come BEFORE importing pyplot
import matplotlib.pyplot as plt
from matplotlib import cm
import scipy.sparse as sp
import scipy.sparse.linalg as spla


# ── Analytical solution and source term ───────────────────────────────────────

def u_exact(x, y):
    return np.sin(np.pi * x) * np.sin(np.pi * y)

def f(x, y):
    return 2.0 * np.pi**2 * np.sin(np.pi * x) * np.sin(np.pi * y)


# ── Matrix and RHS assembly ────────────────────────────────────────────────────

def create_matrix(h):
    """
    Build A_h = (1/h²) * (T ⊗ I + I ⊗ T)
    where T is the 1-D tridiagonal FD matrix.
    This Kronecker construction is correct for ALL valid h.
    """
    n = int(round(1.0 / h)) - 1   # interior points per dimension
    inv_h2 = 1.0 / h**2

    # 1-D tridiagonal: diag(2) + off-diag(-1)
    T = sp.diags([-1, 2, -1], [-1, 0, 1], shape=(n, n), format='csr', dtype=float)

    I = sp.eye(n, format='csr')

    # 2-D operator via Kronecker sum
    A = inv_h2 * (sp.kron(I, T, format='csr') + sp.kron(T, I, format='csr'))
    return A


def create_right_hand_side(h):
    """
    Assemble the RHS vector q_h = f evaluated at interior grid points.
    """
    n = int(round(1.0 / h)) - 1
    b = np.zeros(n * n)
    for j in range(n):          # y-index
        for i in range(n):      # x-index
            x = (i + 1) * h
            y = (j + 1) * h
            b[j * n + i] = f(x, y)
    return b


# ── Error computation ──────────────────────────────────────────────────────────

def compute_error(u_numerical, h):
    """
    Discrete L²-norm (eq. 1.18, Knabner/Angermann) of the error
    ||u_exact - u_h||_{L²_h} = h * sqrt( sum_{i,j} |e_{i,j}|^2 ).
    """
    n = int(round(1.0 / h)) - 1
    err = np.zeros(n * n)
    for j in range(n):
        for i in range(n):
            x = (i + 1) * h
            y = (j + 1) * h
            err[j * n + i] = u_exact(x, y) - u_numerical[j * n + i]
    return h * np.sqrt(np.sum(err**2))


# ── Plot ───────────────────────────────────────────────────────────────────────

def plot_solution(h, u_numerical):
    """
    3-D surface plots of the numerical and analytical solutions.
    """
    n = int(round(1.0 / h)) - 1

    # Embed interior values into full grid (with boundary zeros)
    u_grid = np.zeros((n + 2, n + 2))
    for j in range(n):
        for i in range(n):
            u_grid[j + 1, i + 1] = u_numerical[j * n + i]

    x = np.linspace(0, 1, n + 2)
    y = np.linspace(0, 1, n + 2)
    X, Y = np.meshgrid(x, y)
    U_anal = u_exact(X, Y)

    fig, axes = plt.subplots(
        1, 2, figsize=(12, 5),
        subplot_kw={'projection': '3d'}
    )

    axes[0].plot_surface(X, Y, u_grid, cmap=cm.viridis)
    axes[0].set_title(f'Numerical solution  (h = 1/{int(round(1/h))})')
    axes[0].set_xlabel('x'); axes[0].set_ylabel('y'); axes[0].set_zlabel('u')

    axes[1].plot_surface(X, Y, U_anal, cmap=cm.plasma)
    axes[1].set_title('Analytical solution')
    axes[1].set_xlabel('x'); axes[1].set_ylabel('y'); axes[1].set_zlabel('u')

    plt.tight_layout()
    plt.savefig('poisson_solution.png', dpi=150, bbox_inches='tight')
    #plt.show()


# ── EOC study ─────────────────────────────────────────────────────────────────

hs     = [1.0 / 2**k for k in range(1, 8)]   # h = 1/2, 1/4, ..., 1/128
errors = []

print(f"{'h':>10}  {'||e||_h':>14}")
print("-" * 28)
for h in hs:
    A   = create_matrix(h)
    b   = create_right_hand_side(h)
    u_h = spla.spsolve(A, b)
    err = compute_error(u_h, h)
    errors.append(err)
    print(f"1/{int(round(1/h)):<6}   {err:.6e}")

print(f"\n{'h':>10}  {'EOC':>8}")
print("-" * 22)
for k in range(1, len(errors)):
    eoc = np.log(errors[k-1] / errors[k]) / np.log(2.0)
    print(f"1/{int(round(1/hs[k])):<6}   {eoc:.4f}")

# ── Solution plot for h = 1/64 ────────────────────────────────────────────────
h_plot  = 1.0 / 64
A_plot  = create_matrix(h_plot)
b_plot  = create_right_hand_side(h_plot)
u_plot  = spla.spsolve(A_plot, b_plot)
plot_solution(h_plot, u_plot)

         h         ||e||_h
----------------------------
1/2        1.168503e-01
1/4        2.651464e-02
1/8        6.475373e-03
1/16       1.609482e-03
1/32       4.017888e-04
1/64       1.004109e-04
1/128      2.510046e-05

         h       EOC
----------------------
1/4        2.1398
1/8        2.0338
1/16       2.0084
1/32       2.0021
1/64       2.0005
1/128      2.0001


### Task 2 (3+2+5 P)

 Consider the fourth order boundary value problem:
 Find $u \in C^4(\Omega) \cap C^1(\bar \Omega)$ such that

 \begin{align*}
   \Delta^2 u - \Delta u &= f \qquad \text{ in } \Omega\, ,\\
   \nabla u \cdot n = u &= 0 \qquad\text{ on } \partial\Omega
 \end{align*}

 with $n$ being the outwards pointing normal and $\Omega$ being a Lipschitz domain.

  a) Choose $V$ and derive a weak (variational) formulation: *Find $u\in V$ such that $a(u,v) = b(v) \quad \forall v \in V$.*

**Choice of function space.** Since the boundary conditions require both $u = 0$ and $\nabla u \cdot n = 0$ on $\partial\Omega$, we choose

$$
V = H^2_0(\Omega) := \overline{C_0^\infty(\Omega)}^{\|\cdot\|_{H^2(\Omega)}},
$$
i.e., the closure of smooth compactly supported functions in the $H^2$-norm. Functions in $V$ satisfy $u|_{\partial\Omega} = 0$ and $\nabla u \cdot n|_{\partial\Omega} = 0$ in the trace sense.

**Derivation.** Multiply the strong form by a test function $v \in V$ and integrate over $\Omega$:
$$
\int_\Omega (\Delta^2 u - \Delta u)\, v\, dx = \int_\Omega f\, v\, dx.
$$

**Term $-\Delta u$:** Apply Green's first identity,
$$
-\int_\Omega \Delta u\, v\, dx = \int_\Omega \nabla u \cdot \nabla v\, dx - \underbrace{\int_{\partial\Omega} (\nabla u \cdot n)\, v\, ds}_{=\,0,\ v|_{\partial\Omega}=0}.
$$

**Term $\Delta^2 u$:** Apply Green's first identity with $w = \Delta u$,
$$
\int_\Omega \Delta^2 u\, v\, dx = -\int_\Omega \nabla(\Delta u) \cdot \nabla v\, dx + \underbrace{\int_{\partial\Omega} \nabla(\Delta u)\cdot n\, v\, ds}_{=\,0,\ v|_{\partial\Omega}=0}.
$$
Apply Green's first identity once more,
$$
-\int_\Omega \nabla(\Delta u)\cdot \nabla v\, dx = \int_\Omega \Delta u\, \Delta v\, dx - \underbrace{\int_{\partial\Omega} \Delta u\, (\nabla v \cdot n)\, ds}_{=\,0,\ \nabla v\cdot n|_{\partial\Omega}=0}.
$$

**Weak formulation.** Find $u \in V = H^2_0(\Omega)$ such that
$$
a(u, v) = b(v) \qquad \forall\, v \in V,
$$
where
$$
\boxed{a(u,v) = \int_\Omega \Delta u\, \Delta v\, dx + \int_\Omega \nabla u \cdot \nabla v\, dx, \qquad b(v) = \int_\Omega f\, v\, dx.}
$$
    
    ---

  
  b) Why does the boundary condition make sense in your $V$?

  Since $\Omega$ is a Lipschitz domain, the **trace operators**
    $$
    \gamma_0 : H^2(\Omega) \to H^{3/2}(\partial\Omega), \qquad \gamma_1 : H^2(\Omega) \to H^{1/2}(\partial\Omega), \quad \gamma_1 u := \nabla u \cdot n|_{\partial\Omega}
    $$
    are well-defined, linear, and continuous. By definition of $H^2_0(\Omega)$ as the closure of $C_0^\infty(\Omega)$,
    $$
    H^2_0(\Omega) = \ker(\gamma_0) \cap \ker(\gamma_1),
    $$
    so both conditions $u|_{\partial\Omega} = 0$ and $\nabla u \cdot n|_{\partial\Omega} = 0$ are meaningful for every $u \in V$ — they hold in the sense of traces. In particular, $V \subset H^1_0(\Omega)$, so the Dirichlet condition is automatically inherited.
    
    ---
  
  c) Show that your weak (variational) formulation allows for a unique solution.
  
  Hints:
  - Proceed as we did in the lecture (Lax-Milgram). You may also use the result of Exercise 1.10 in the lecture notes (Google Drive).
  - You may use without prove that for $u \in H^2_0(\Omega)$
  \begin{equation*}
   \| \Delta u \|_{L^2(\Omega)} = \| D^2 u \|_{L^2(\Omega)} = |u|_{H^2(\Omega)}.
  \end{equation*}


    We verify the four conditions of the **Lax–Milgram theorem** on the Hilbert space $V = H^2_0(\Omega)$ equipped with the norm $\|\cdot\|_{H^2(\Omega)}$.

    We use the hint, which states that for $u \in H^2_0(\Omega)$:
    $$
    \|\Delta u\|_{L^2(\Omega)} = \|D^2 u\|_{L^2(\Omega)} = |u|_{H^2(\Omega)}.
    $$
    From Exercise 1.10, $|\cdot|_{H^2(\Omega)} = \|\Delta\,\cdot\|_{L^2(\Omega)}$ is an **equivalent norm** on $H^2_0(\Omega)$, i.e., there exists $c > 0$ such that
    $$
    c\,\|u\|_{H^2(\Omega)} \leq \|\Delta u\|_{L^2(\Omega)} \qquad \forall\, u \in H^2_0(\Omega). \tag{$*$}
    $$
    
    **1. $V$ is a Hilbert space.**  
    $H^2_0(\Omega)$ is a closed subspace of the Hilbert space $H^2(\Omega)$, hence itself a Hilbert space. ✓
    
    **2. $b$ is a continuous linear functional.**  
    Let $f \in L^2(\Omega)$. By the Cauchy–Schwarz inequality,
    $$
    |b(v)| = \left|\int_\Omega f\, v\, dx\right| \leq \|f\|_{L^2}\|v\|_{L^2} \leq \|f\|_{L^2}\|v\|_{H^2},
    $$
    so $b \in V'$ with $\|b\|_{V'} \leq \|f\|_{L^2}$. ✓
    
    **3. $a$ is bounded (continuous).**  
    By the Cauchy–Schwarz inequality and the continuous embedding $H^2 \hookrightarrow H^1 \hookrightarrow L^2$:
    $$
    |a(u,v)| \leq \|\Delta u\|_{L^2}\|\Delta v\|_{L^2} + \|\nabla u\|_{L^2}\|\nabla v\|_{L^2}
    \leq |u|_{H^2}|v|_{H^2} + |u|_{H^1}|v|_{H^1}
    \leq 2\,\|u\|_{H^2}\|v\|_{H^2}.
    $$
    So $a$ is bounded with constant $M = 2$. ✓
    
    **4. $a$ is coercive.**  
    Using the hint and $(*)$:
    $$
    a(u,u) = \|\Delta u\|_{L^2}^2 + \|\nabla u\|_{L^2}^2 \geq \|\Delta u\|_{L^2}^2 = |u|_{H^2}^2 \geq c^2\,\|u\|_{H^2}^2.
    $$
    So $a$ is coercive with constant $\alpha = c^2 > 0$. ✓
    
    **Conclusion.**  
    Since $V = H^2_0(\Omega)$ is a Hilbert space, $a(\cdot,\cdot)$ is a bounded and coercive bilinear form, and $b(\cdot)$ is a bounded linear functional, the **Lax–Milgram theorem** guarantees the existence of a **unique** solution $u \in H^2_0(\Omega)$ satisfying
    $$
    a(u,v) = b(v) \qquad \forall\, v \in H^2_0(\Omega). \qquad \blacksquare
    $$
